In [29]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("../data/archive/combine.csv",low_memory=False)

In [3]:
df.columns=df.columns.str.strip()

In [4]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,55054,109.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,55055,52.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,46236,34.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,54863,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [5]:
df.shape

(2214469, 79)

In [6]:
df['Label']=df['Label'].str.strip().str.upper()

In [7]:
df['Label'].value_counts()

Label
BENIGN              1672837
DOS HULK             231073
PORTSCAN             158930
DDOS                 128027
DOS GOLDENEYE         10293
DOS SLOWLORIS          5796
DOS SLOWHTTPTEST       5499
BOT                    1966
INFILTRATION             36
HEARTBLEED               11
Name: count, dtype: int64

In [8]:
df['Label']=df['Label'].replace({
    'DOS HULK':'DOS',
    'DOS GOLDENEYE':'DOS',
    'DOS SLOWLORIS':'DOS',
    'DOS SLOWHTTPTEST':'DOS'})

In [9]:
df=df[df['Label'].isin(['BENIGN','DOS','DDOS','PORTSCAN'])].copy()

In [10]:
print("Class distribution after Label mapping and filtering",df['Label'].value_counts())

Class distribution after Label mapping and filtering Label
BENIGN      1672837
DOS          252661
PORTSCAN     158930
DDOS         128027
Name: count, dtype: int64


In [11]:
print("Shape before cleaning",df.shape)

Shape before cleaning (2212455, 79)


In [12]:
df.replace([np.inf,-np.inf],np.nan,inplace=True)

In [13]:
df.dropna(inplace=True)

In [14]:
df.drop_duplicates(inplace=True)

In [15]:
df.reset_index(drop=True,inplace=True)

In [16]:
print("Shape after cleaning",df.shape)

Shape after cleaning (1939657, 79)


In [17]:
x=df.drop(['Label'],axis=1)
y=df['Label']

In [18]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,y,
    test_size=0.3,
    random_state=42,
    stratify=y)

In [19]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [20]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced")

In [21]:
model.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [24]:
y_pred=model.predict(x_test)

In [26]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

print("Accuracy:",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred,digits=6))
print(confusion_matrix(y_test,y_pred))

Accuracy: 0.9581301190242963
              precision    recall  f1-score   support

      BENIGN   0.998903  0.948114  0.972846    458161
        DDOS   0.896603  0.998672  0.944889     38405
         DOS   0.889637  0.991467  0.937795     58124
    PORTSCAN   0.688054  0.998346  0.814654     27208

    accuracy                       0.958130    581898
   macro avg   0.868299  0.984150  0.917546    581898
weighted avg   0.966703  0.958130  0.960103    581898

[[434389   4361   7099  12312]
 [    20  38354     28      3]
 [   434     62  57628      0]
 [    23      0     22  27163]]


In [28]:
import joblib

joblib.dump(model,"../models/logistic_regression_model.pkl")
joblib.dump(scaler,"../models/logistic_scaler.pkl")

['../models/logistic_scaler.pkl']